# P45 — Destilar el conocimiento de una red neuronal

## 1. Título y paper

**Paper:** *Distilling the Knowledge in a Neural Network*  
**Autoría:** Geoffrey Hinton, Oriol Vinyals, Jeff Dean  
**Año y venue:** 2015 · arXiv:1503.02531  
**Nivel:** L2 · **Motor:** `distillation`  
**Ficha completa:** [`P45_distillation`](../../papers/foundational/P45_distillation/README.md)

**Hito:** Las probabilidades del maestro contienen más información que la etiqueta correcta: el modelo pequeño aprende de esa estructura.

- [arXiv:1503.02531](https://arxiv.org/abs/1503.02531)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los modelos grandes o los conjuntos de modelos daban los mejores resultados pero eran caros de servir, y entrenar el modelo pequeño con las etiquetas duras daba mucho peor resultado.
2. Ejecutar una implementación mínima de la propuesta: Entrenar el modelo pequeño para reproducir la distribución completa del maestro, suavizada con una temperatura que revela la estructura de similitud entre clases.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P04


## 4. Intuición

Un examen tipo test corregido solo con «bien/mal» enseña menos que uno donde el profesor te dice qué otras respuestas estuvieron cerca de ser correctas. Esa información extra es lo que el maestro le pasa al alumno.


## 5. Concepto mínimo

```text
Etiqueta dura :  perro=1, lobo=0, gato=0, coche=0
Objetivo suave:  perro=0.6, lobo=0.25, gato=0.13, coche=0.02   ← con temperatura T

    p_i = softmax(z_i / T)      T alta → distribución más informativa
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('distillation', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué información contiene el objetivo suave que la etiqueta dura no?
2. ¿Qué le pasa a la entropía al subir T?
3. ¿Por qué eso ayuda a un modelo pequeño?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('distillation', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('distillation', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

El maestro dice que un perro se parece más a un lobo que a un gato, y muchísimo más que a un coche. Esa **estructura de similitud entre clases** es conocimiento que la etiqueta dura tira a la basura, y es lo que permite al alumno aprender con muchos menos datos.


## 10. Comentario pedagógico

La destilación es hoy la razón de que existan modelos pequeños sorprendentemente buenos, incluidos los de razonamiento de [P22](../../papers/foundational/P22_deepseek_r1/README.md). El alumno hereda comportamiento — incluidos los errores del maestro.


## 11. Error o anti-patrón deliberado

Anti-patrón: destilar de un maestro sin evaluar al maestro.


In [ ]:
print('El alumno aprende la distribucion del maestro, sesgos y errores incluidos.')
print('Si el maestro confunde sistematicamente dos clases, el alumno lo heredara')
print('y ademas con mas confianza, porque lo aprendio como objetivo suave.')

## 12. Corrección

Lo que hay que comprobar antes de destilar:


In [ ]:
checklist = {'maestro evaluado': 'en el mismo conjunto donde se medira al alumno',
             'errores caracterizados': 'que clases confunde y con que frecuencia',
             'temperatura': 'elegida en validacion, no copiada de un paper',
             'alumno evaluado aparte': 'nunca solo contra el maestro'}
show(checklist)

## 13. Desafío guiado

Sube la temperatura a 20 y observa si la distribución sigue siendo informativa o se vuelve uniforme.


In [ ]:
r = run_paper_lab('distillation', seed=3)['result']
show(r)

## 14. Desafío autónomo

Destila un modelo pequeño a partir de uno mayor en una tarea de clasificación. Compara con entrenar el pequeño solo con etiquetas duras, y mide también si hereda los errores del maestro.


## 15. Evidencia de aprendizaje

Guarda la tabla de distribuciones por temperatura, la entropía y tu checklist previo a destilar.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P45_distillation/README.md) · evaluación formal: [`assessments/papers/P45_distillation.md`](../../assessments/papers/P45_distillation.md)


## 16. Cierre

Modelos pequeños que heredan capacidad. Ahora, un cambio de arquitectura que nadie esperaba en visión.


## 17. Conexión con el siguiente hito

- P22
- modelos pequeños de razonamiento

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
